## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

from mingpt.dataset_v2 import CharDataset, CharTokenizer, make_datasets
from mingpt.model_v2 import GPT, GPTConfig
from mingpt.trainer_v2 import Trainer, TrainerConfig, make_loaders, train_on_toy_dataset, sample_shakespeare

In [2]:
# set up logging
import logging
logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s -   %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
)
logger = logging.getLogger("mingpt")
logger.setLevel(logging.INFO)

In [3]:
# make deterministic
from mingpt.utils import set_seed
set_seed(42)

## 01 Run on a toy dataset (1)

In [4]:
# Load text 
text = open('input.txt', 'r').read()

# Create tokenizer
tokenizer = CharTokenizer(text)

# Split into train and validation sets
train_ds_toy, val_ds_toy = make_datasets(
    text=text[:128*10], 
    tokenizer=tokenizer,
    val_fraction=0.2
    )

# Create configuration for the model
model_config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    block_size=train_ds_toy.block_size,
)

# Create the model
model = GPT(model_config)

# Create trainer config
trainer_config = TrainerConfig(
    lr_decay=False
)

# Create the trainer
trainer = Trainer(
    model=model,
    train_dataset=train_ds_toy,
    val_dataset=val_ds_toy,
    config=trainer_config
    )

11/03/2026 10:07:09 - INFO - mingpt.dataset_v2 -   Training dataset with 896 characters
11/03/2026 10:07:09 - INFO - mingpt.dataset_v2 -   Validation dataset with 128 characters
11/03/2026 10:07:09 - INFO - mingpt.model_v2 -   number of parameters: 25.35M
11/03/2026 10:07:09 - INFO - mingpt.trainer_v2 -   Created train loader with 14 batches
11/03/2026 10:07:09 - INFO - mingpt.trainer_v2 -   Created val loader with 2 batches


In [5]:
trainer.train()

  0%|          | 0/14 [00:00<?, ?it/s]/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
epoch 1 iter 13: train loss 2.93993. lr 3.000000e-04: 100%|██████████| 14/14 [00:39<00:00,  2.79s/it]
11/03/2026 10:08:06 - INFO - mingpt.trainer_v2 -   val loss: 3.1762
11/03/2026 10:08:06 - INFO - mingpt.trainer_v2 -   saving checkpoints/ckpt.pt
epoch 2 iter 13: train loss 2.35306. lr 3.000000e-04: 100%|██████████| 14/14 [00:42<00:00,  3.01s/it]
11/03/2026 10:08:50 - INFO - mingpt.trainer_v2 -   val loss: 2.6073
11/03/2026 10:08:50 - INFO - mingpt.trainer_v2 -   saving checkpoints/ckpt.pt


In [6]:
# Alright, let's sample some character-level Shakespeare
from mingpt.utils import sample

context = "O God, O God!"
x = torch.tensor([tokenizer.stoi[s] for s in context], dtype=torch.long)[None,...].to(trainer.device)
y = sample(model, x, 2000, temperature=1.0, sample=True, top_k=10)[0]
completion = ''.join([tokenizer.itos[int(i)] for i in y])
print(completion)

O God, O God!
Fitis izer ptowan:ore kne cer we coicitikingore t ce athe ize titicar collit the co ous oure usdorsoren:.
W'comizen amie our pe t g we we tod
Ar Cis.


Alve the ize uthe ure ar Cizurs,irened ore tar ure t peveit.
Fe pe thachane toure owe he thus we t, udity t, akf the wodo od ullisornd t, owo woure theve the ayhhe thawanoy war auy ous whithileicorin:


F wened gelledyo thithen we Ciulansur ticasod.
Alld

Ficorun at irere ar Cie thand atharir are whomithe t, andarentize pun centold wy hutit irs ouns wold ato pe t toouct oizo ou titheiris uno tor pe ounoutheie tize t utoushe thes the ay utoizoneresout ouyhe ty
s anthar iusour wowe the an ane wodhe gold pithere poleamentoizeathevere ar we on thicon:

W the at iste ppe amedanod


Alllves wor angod ty weansuce aresithowesd it tize anel thawoy
Wonsd t't the alllllves, therenthicharyize the heryithik hhenoll herisouned there pisd the uryhed;


Sangho awowesol:
Fitharfnodizel ical: ge t knther thirere the t.
Wey, pesounsusures hu

## 02 Run on a toy dataset (2)

In [4]:
# Training on a toy dataset
train_on_toy_dataset()

11/03/2026 10:22:34 - INFO - mingpt.dataset_v2 -   Training dataset with 896 characters
11/03/2026 10:22:34 - INFO - mingpt.dataset_v2 -   Validation dataset with 128 characters
11/03/2026 10:22:34 - INFO - mingpt.model_v2 -   number of parameters: 25.35M
11/03/2026 10:22:34 - INFO - mingpt.trainer_v2 -   Created train loader with 14 batches
11/03/2026 10:22:34 - INFO - mingpt.trainer_v2 -   Created val loader with 2 batches
  0%|          | 0/14 [00:00<?, ?it/s]/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
epoch 1 iter 13: train loss 2.93993. lr 3.000000e-04: 100%|██████████| 14/14 [00:40<00:00,  2.91s/it]
11/03/2026 10:23:16 - INFO - mingpt.trainer_v2 -   val loss: 3.1762
11/03/2026 10:23:16 - INFO - mingpt.trainer_v2 -   saving checkpoints/ckpt



===Sampling from the trained model:===
O God, O God!
Fitis izer ptowan:ore kne cer we coicitikingore t ce athe ize titicar collit the co ous oure usdorsoren:.
W'comizen amie our pe t g we we tod
Ar Cis.


Alve the ize uthe ure ar Cizurs,irened ore tar ure t peveit.
Fe pe thachane toure owe he thus we t, udity t, akf the wodo od ullisornd t, owo woure theve the ayhhe thawanoy war auy ous whithileicorin:


F wened gelledyo thithen we Ciulansur ticasod.
Alld

Ficorun at irere ar Cie thand atharir are whomithe t, andarentize pun centold wy hutit irs ouns wold ato pe t toouct oizo ou titheiris uno tor pe ounoutheie tize t utoushe thes the ay utoizoneresout ouyhe ty
s anthar iusour wowe the an ane wodhe gold pithere poleamentoizeathevere ar we on thicon:

W the at iste ppe amedanod


Alllves wor angod ty weansuce aresithowesd it tize anel thawoy
Wonsd t't the alllllves, therenthicharyize the heryithik hhenoll herisouned there pisd the uryhed;


Sangho awowesol:
Fitharfnodizel ical: ge t kn